# Tugas Mandiri Data Wrangling – Pertemuan 2
**Analisis Kampanye Kupon Diskon Akhir Bulan – PT Nusantara Retail Mandiri**

| | |
|---|---|
| Nama Mahasiswa | Lyokta Nur Latifah |
| NIM | 25200010 |
| Peran | Associate Data Engineer / Data Wrangler |

**Kebutuhan bisnis (Head of Business Strategy):** *berapa total transaksi yang menggunakan kupon, bagaimana profil loyalitas pelanggan yang memakainya, dan apakah pesanan tersebut sudah terkirim oleh mitra logistik?*

# BAGIAN A – Analisis Konseptual & Tata Kelola Data

## A.1 Pemetaan Kebutuhan Informasi Bisnis → Kebutuhan Teknis Data

| No | Kebutuhan Informasi Bisnis | Atribut Data | Tipe Data | Sumber Asal | Peran Atribut |
|---|---|---|---|---|---|
| 1 | Total transaksi berkupon | `order_id` | string (VARCHAR) | CSV POS (`transaksi_pos.csv`) | Primary key transaksi |
| 2 | Total transaksi berkupon | `coupon_code` | string, nullable | CSV POS | Penanda pemakaian kupon (NULL = tanpa kupon) |
| 3 | Total transaksi berkupon | `total_amount` | integer / decimal (Rupiah) | CSV POS | Ukuran nilai transaksi (dijumlahkan) |
| 4 | Periode kampanye akhir bulan | `order_date` | datetime64 | CSV POS | Filter periode, validasi *future date* |
| 5 | Penghubung ke pelanggan | `customer_id` | string | CSV POS & DB CRM | Foreign key / join key ke CRM |
| 6 | Profil loyalitas pelanggan | `membership_tier` | string (kategori: Bronze/Silver/Gold/Platinum) | DB CRM (`customers`) | Segmentasi loyalitas |
| 7 | Profil pelanggan | `customer_name`, `city` | string | DB CRM | Identitas & demografi |
| 8 | Pelanggan valid | `is_active` | integer (0/1) | DB CRM | Filter pelanggan aktif (`WHERE is_active = 1`) |
| 9 | Status pengiriman | `order_id` | string | REST API logistik | Join key ke transaksi |
| 10 | Status pengiriman | `delivery_status` | string (Delivered / In Transit / Pending Pickup / Returned) | REST API logistik | Jawaban "sudah terkirim?" |
| 11 | Pelacakan paket | `tracking_number` | string | REST API logistik | Identitas paket |
| 12 | Detail kurir & tujuan | `courier.name`, `courier.service`, `destination.city`, `destination.province`, `timeline.*` | string / date | REST API logistik (nested JSON, diratakan dengan `json_normalize`) | Atribut pendukung analisis |

## A.2 Matriks RACI

Aktivitas: **(1)** Menetapkan definisi bisnis (apa itu "transaksi berkupon", "terkirim"); **(2)** Menetapkan standar kualitas & kamus data; **(3)** Membangun pipeline ingestion & integrasi; **(4)** Validasi kualitas data (assertion); **(5)** Analisis kinerja kampanye; **(6)** Persetujuan akses & penggunaan data.

| Aktivitas | Head of Business Strategy (Data Owner) | Data Management Office / Data Steward | Tim Data Wrangling (Data Custodian / Engineer) | BI Analyst (Data User) |
|---|---|---|---|---|
| 1. Definisi bisnis & KPI kampanye | **A / R** | C | C | C |
| 2. Standar kualitas data & kamus data | A | **R** | C | I |
| 3. Pipeline ingestion & integrasi multi-sumber | I | C | **A / R** | C |
| 4. Validasi kualitas & integritas (assertion) | I | **A** | **R** | I |
| 5. Analisis kinerja kampanye kupon | **A** | I | C | **R** |
| 6. Persetujuan akses & penggunaan data | **A** | **R** | C | I |

*Keterangan:* **R** = Responsible (mengerjakan), **A** = Accountable (penanggung jawab akhir, hanya satu per aktivitas), **C** = Consulted (dimintai masukan), **I** = Informed (diberi informasi).
Pada aktivitas 1, 5, dan 6 Data Owner memegang akuntabilitas karena keputusan bisnis ada padanya. Pada aktivitas 2 dan 4 Data Steward akuntabel atas kepatuhan standar kualitas, sedangkan Data Custodian yang mengeksekusi secara teknis.

# BAGIAN B – Komputasi Python & Data Ingestion Pipeline

In [1]:
import os, json, threading
from http.server import BaseHTTPRequestHandler, HTTPServer

import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Bangkitkan dataset dummy bila belum ada (lihat generate_dummy_data.py)
if not all(os.path.exists(f) for f in ["transaksi_pos.csv", "crm_loyalty.db", "logistik_mock.json"]):
    import generate_dummy_data
print("Dataset siap digunakan.")

Dataset siap digunakan.


## B.1 Ekstraksi Berkas File (CSV)
Kolom ID (`order_id`, `customer_id`) dipaksa bertipe `str` agar tidak kehilangan format/nol di depan, dan `order_date` di-parse menjadi `datetime64`.

In [2]:
print("[INFO] Mengimpor berkas transaksi POS...")
df_transaksi = pd.read_csv(
    "transaksi_pos.csv",
    dtype={"order_id": str, "customer_id": str},
    parse_dates=["order_date"],
)
print(df_transaksi.shape)
print(df_transaksi.dtypes)
df_transaksi.head()

[INFO] Mengimpor berkas transaksi POS...
(500, 6)
order_id                   str
customer_id                str
order_date      datetime64[us]
store_id                   str
coupon_code                str
total_amount             int64
dtype: object


,order_id,customer_id,order_date,store_id,coupon_code,total_amount
0,ORD20260801,CUST0120,2026-08-10,TKO-BDG,NaN,543000
1,ORD20260802,CUST0055,2026-08-25,ONLINE,NaN,362000
2,ORD20260803,GUEST007,2026-09-20,ONLINE,NaN,737000
3,ORD20260804,CUST0009,2026-08-14,TKO-JKT,NaN,899000
4,ORD20260805,CUST0096,2026-09-26,TKO-BDG,NaN,722000


## B.2 Koneksi & Kueri Database (SQL)
Menggunakan SQLite melalui SQLAlchemy. Hanya pelanggan aktif (`WHERE is_active = 1`) yang diambil.

In [3]:
print("[INFO] Menghubungkan ke database CRM...")
engine = create_engine("sqlite:///crm_loyalty.db")

query_crm = """
    SELECT customer_id, customer_name, membership_tier, city
    FROM customers
    WHERE is_active = 1
"""
df_crm = pd.read_sql(query_crm, con=engine)

total_crm = pd.read_sql("SELECT COUNT(*) AS n FROM customers", con=engine).iloc[0, 0]
print(f"Pelanggan aktif: {len(df_crm)} dari total {total_crm} di tabel customers")
df_crm.head()

[INFO] Menghubungkan ke database CRM...
Pelanggan aktif: 103 dari total 120 di tabel customers


,customer_id,customer_name,membership_tier,city
0,CUST0002,Rizky Hidayat,Silver,Pemalang
1,CUST0003,Wulan Rahmawati,Bronze,Surabaya
2,CUST0004,Siti Kusuma,Silver,Denpasar
3,CUST0005,Rina Santoso,Bronze,Yogyakarta
4,CUST0006,Nanda Setiawan,Silver,Surabaya


## B.3 Konsumsi REST API & Normalisasi JSON
Sandbox pengerjaan tidak selalu bisa mengakses internet publik, sehingga dibuat **mock REST API lokal** (server HTTP di `127.0.0.1`) yang menyajikan berkas `logistik_mock.json` pada endpoint `/mocks/dummy-logistik/deliveries`. Request tetap dilakukan sungguhan dengan `requests.get()`. Untuk memakai API publik, cukup ganti nilai `api_url`.

In [4]:
# --- Mock REST API lokal (menggantikan endpoint ekspedisi pihak ketiga) ---
class MockLogistikHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path.startswith("/mocks/dummy-logistik/deliveries"):
            with open("logistik_mock.json", "rb") as f:
                body = f.read()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_response(404); self.end_headers()
    def log_message(self, *args):  # senyapkan log server
        pass

server = HTTPServer(("127.0.0.1", 0), MockLogistikHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
api_url = f"http://127.0.0.1:{server.server_port}/mocks/dummy-logistik/deliveries"

# --- Request GET ---
print("[INFO] Menarik data pengiriman dari REST API...")
response = requests.get(api_url, timeout=10)
print("HTTP status:", response.status_code)

if response.status_code == 200:
    data_logistik = response.json()
    print("Kunci level atas JSON:", list(data_logistik.keys()))
    print("Contoh 1 record (nested):")
    print(json.dumps(data_logistik["results"][0], indent=2))

    # Ratakan (flatten) JSON bersarang -> kolom 'courier.name', 'destination.city', dst.
    df_logistik = pd.json_normalize(data_logistik["results"])
else:
    raise RuntimeError("[ERROR] Gagal memanggil API")

# Samakan nama kolom hasil flatten agar rapi (titik -> underscore)
df_logistik.columns = df_logistik.columns.str.replace(".", "_", regex=False)
df_logistik["order_id"] = df_logistik["order_id"].astype(str)
print(df_logistik.shape)
df_logistik.head()

[INFO] Menarik data pengiriman dari REST API...
HTTP status: 200
Kunci level atas JSON: ['status', 'count', 'results']
Contoh 1 record (nested):
{
  "order_id": "ORD20260801",
  "tracking_number": "JNX2443503916",
  "delivery_status": "Returned",
  "courier": {
    "name": "JNE",
    "service": "REG"
  },
  "destination": {
    "city": "Denpasar",
    "province": "Bali"
  },
  "timeline": {
    "shipped_at": "2026-09-16",
    "delivered_at": null
  }
}
(460, 9)


,order_id,tracking_number,delivery_status,courier_name,courier_service,destination_city,destination_province,timeline_shipped_at,timeline_delivered_at
0,ORD20260801,JNX2443503916,Returned,JNE,REG,Denpasar,Bali,2026-09-16,NaN
1,ORD20260802,JNX1441666429,Delivered,SiCepat,YES,Semarang,Jawa Tengah,2026-08-24,2026-09-22
2,ORD20260803,JNX7481944261,In Transit,JNE,YES,Medan,Sumatera Utara,2026-08-23,NaN
3,ORD20260804,JNX6455767859,Delivered,SiCepat,ECO,Denpasar,Bali,2026-08-03,2026-08-24
4,ORD20260805,JNX7111464006,Pending Pickup,JNE,REG,Medan,Sumatera Utara,2026-08-17,NaN


## B.4 Integrasi Data Multi-Sumber (Merging)

**Pilihan join dan alasan logisnya**

1. **`df_transaksi` ⟕ `df_crm` → `LEFT JOIN` pada `customer_id`.**
   Tabel transaksi adalah *fakta utama* karena pertanyaan bisnis berawal dari "berapa total transaksi berkupon". Jika memakai `INNER JOIN`, transaksi dari pelanggan non-member (mis. `GUEST…`) atau pelanggan non-aktif akan **hilang diam-diam**, sehingga total transaksi dan total omzet kampanye **terhitung lebih rendah dari kenyataan**. Dengan `LEFT JOIN`, semua transaksi tetap terhitung, dan pelanggan tanpa profil CRM aktif ditandai sebagai "Non-Member / Tidak Aktif" pada analisis loyalitas.
2. **Hasil ⟕ `df_logistik` → `LEFT JOIN` pada `order_id`.**
   Pesanan yang **belum diserahkan ke kurir** tidak ada di sistem ekspedisi. Itu justru informasi penting ("belum terkirim"). `INNER JOIN` akan membuang pesanan tersebut sehingga tingkat pengiriman tampak lebih baik daripada kenyataan.

Setelah merge dilakukan pemeriksaan jumlah baris: karena `order_id` unik di tabel logistik dan `customer_id` unik di CRM, jumlah baris `df_master_analisis` harus **sama** dengan `df_transaksi` (tidak ada duplikasi akibat *fan-out*).

In [5]:
print("[INFO] Melakukan integrasi data multi-sumber...")
n_awal = len(df_transaksi)

df_master_analisis = (
    df_transaksi
    .merge(df_crm, on="customer_id", how="left", validate="m:1")
    .merge(df_logistik, on="order_id", how="left", validate="1:1")
)

print(f"Baris transaksi awal   : {n_awal}")
print(f"Baris df_master_analisis: {len(df_master_analisis)}")
assert len(df_master_analisis) == n_awal, "Jumlah baris berubah setelah merge (kemungkinan duplikasi)!"
print(df_master_analisis.shape)
df_master_analisis.head()

[INFO] Melakukan integrasi data multi-sumber...
Baris transaksi awal   : 500
Baris df_master_analisis: 500
(500, 17)


,order_id,customer_id,order_date,store_id,coupon_code,total_amount,customer_name,membership_tier,city,tracking_number,delivery_status,courier_name,courier_service,destination_city,destination_province,timeline_shipped_at,timeline_delivered_at
0,ORD20260801,CUST0120,2026-08-10,TKO-BDG,NaN,543000,Putri Kusuma,Bronze,Makassar,JNX2443503916,Returned,JNE,REG,Denpasar,Bali,2026-09-16,NaN
1,ORD20260802,CUST0055,2026-08-25,ONLINE,NaN,362000,Hadi Nugroho,Bronze,Medan,JNX1441666429,Delivered,SiCepat,YES,Semarang,Jawa Tengah,2026-08-24,2026-09-22
2,ORD20260803,GUEST007,2026-09-20,ONLINE,NaN,737000,NaN,NaN,NaN,JNX7481944261,In Transit,JNE,YES,Medan,Sumatera Utara,2026-08-23,NaN
3,ORD20260804,CUST0009,2026-08-14,TKO-JKT,NaN,899000,Nanda Wijaya,Bronze,Denpasar,JNX6455767859,Delivered,SiCepat,ECO,Denpasar,Bali,2026-08-03,2026-08-24
4,ORD20260805,CUST0096,2026-09-26,TKO-BDG,NaN,722000,Rizky Kusuma,Gold,Semarang,JNX7111464006,Pending Pickup,JNE,REG,Medan,Sumatera Utara,2026-08-17,NaN


### Jawaban Kebutuhan Bisnis (hasil dari `df_master_analisis`)
Transaksi berkupon = baris dengan `coupon_code` tidak kosong.

In [6]:
df_kupon = df_master_analisis[df_master_analisis["coupon_code"].notna()].copy()

# 1) Total transaksi berkupon
print("=== 1. TOTAL TRANSAKSI BERKUPON ===")
print(f"Jumlah transaksi berkupon : {len(df_kupon)} dari {len(df_master_analisis)} "
      f"({len(df_kupon)/len(df_master_analisis):.1%})")
print(f"Total nilai transaksi     : Rp {df_kupon['total_amount'].sum():,.0f}")
print()
display(df_kupon.groupby("coupon_code")["total_amount"]
        .agg(jumlah_transaksi="count", total_nilai="sum").sort_values("jumlah_transaksi", ascending=False))

# 2) Profil loyalitas
print("=== 2. PROFIL LOYALITAS PELANGGAN PEMAKAI KUPON ===")
df_kupon["membership_tier"] = df_kupon["membership_tier"].fillna("Non-Member / Tidak Aktif")
profil = (df_kupon.groupby("membership_tier")
          .agg(jumlah_transaksi=("order_id", "count"),
               pelanggan_unik=("customer_id", "nunique"),
               total_nilai=("total_amount", "sum")))
profil["persentase_transaksi"] = (profil["jumlah_transaksi"] / profil["jumlah_transaksi"].sum() * 100).round(1)
display(profil.sort_values("jumlah_transaksi", ascending=False))

# 3) Status pengiriman
print("=== 3. STATUS PENGIRIMAN PESANAN BERKUPON ===")
df_kupon["delivery_status"] = df_kupon["delivery_status"].fillna("Belum diserahkan ke kurir")
kirim = df_kupon["delivery_status"].value_counts().rename("jumlah_pesanan").to_frame()
kirim["persentase"] = (kirim["jumlah_pesanan"] / kirim["jumlah_pesanan"].sum() * 100).round(1)
display(kirim)

=== 1. TOTAL TRANSAKSI BERKUPON ===
Jumlah transaksi berkupon : 199 dari 500 (39.8%)
Total nilai transaksi     : Rp 153,174,000



,jumlah_transaksi,total_nilai
coupon_code,,
AKHIRBULAN20,98,76599000
GAJIAN15,62,48453000
HEMAT10,39,28122000


=== 2. PROFIL LOYALITAS PELANGGAN PEMAKAI KUPON ===


,jumlah_transaksi,pelanggan_unik,total_nilai,persentase_transaksi
membership_tier,,,,
Bronze,70,36,45170000,35.2
Gold,38,19,33611000,19.1
Silver,36,20,30144000,18.1
Non-Member / Tidak Aktif,35,20,27734000,17.6
Platinum,20,8,16515000,10.1


=== 3. STATUS PENGIRIMAN PESANAN BERKUPON ===


,jumlah_pesanan,persentase
delivery_status,,
Delivered,111,55.8
In Transit,53,26.6
Belum diserahkan ke kurir,17,8.5
Pending Pickup,9,4.5
Returned,9,4.5


# BAGIAN C – Metadata, Kamus Data, & Validasi Aturan Bisnis

## C.1 Otomasi Kamus Data
Fungsi `generate_data_dictionary()` menghasilkan nama kolom, tipe data, jumlah & persentase *missing value*, jumlah nilai unik, dan sampel nilai (hingga 3 nilai unik non-null).

In [7]:
def generate_data_dictionary(df: pd.DataFrame, n_sample: int = 3) -> pd.DataFrame:
    """Menghasilkan ringkasan metadata (kamus data) dari sebuah DataFrame."""
    rows = []
    for col in df.columns:
        n_missing = int(df[col].isnull().sum())
        sampel = df[col].dropna().unique()[:n_sample]
        rows.append({
            "Nama Kolom": col,
            "Tipe Data (dtype)": str(df[col].dtype),
            "Jumlah Missing": n_missing,
            "Missing (%)": round(n_missing / len(df) * 100, 2),
            "Jumlah Nilai Unik": int(df[col].nunique(dropna=True)),
            "Sampel Nilai": " | ".join(str(v) for v in sampel) if len(sampel) else "-",
        })
    return pd.DataFrame(rows)

kamus_df = generate_data_dictionary(df_master_analisis)
kamus_df.to_csv("kamus_data_hasil.csv", index=False)
print("[INFO] Kamus data berhasil disimpan ke kamus_data_hasil.csv")
kamus_df

[INFO] Kamus data berhasil disimpan ke kamus_data_hasil.csv


,Nama Kolom,Tipe Data (dtype),Jumlah Missing,Missing (%),Jumlah Nilai Unik,Sampel Nilai
0,order_id,str,0,0.0,500,ORD20260801 | ORD20260802 | ORD20260803
1,customer_id,str,0,0.0,129,CUST0120 | CUST0055 | GUEST007
2,order_date,datetime64[us],0,0.0,60,2026-08-10 00:00:00 | 2026-08-25 00:00:00 | 20...
3,store_id,str,0,0.0,4,TKO-BDG | ONLINE | TKO-JKT
4,coupon_code,str,301,60.2,3,AKHIRBULAN20 | GAJIAN15 | HEMAT10
5,total_amount,int64,0,0.0,417,543000 | 362000 | 737000
6,customer_name,str,95,19.0,82,Putri Kusuma | Hadi Nugroho | Nanda Wijaya
7,membership_tier,str,95,19.0,4,Bronze | Gold | Silver
8,city,str,95,19.0,10,Makassar | Medan | Denpasar
9,tracking_number,str,40,8.0,460,JNX2443503916 | JNX1441666429 | JNX7481944261


## C.2 Validasi Kualitas & Integritas (Automated Assertions)
Tiga aturan wajib dari soal, ditambah beberapa pemeriksaan tambahan (kunci tidak null, integritas join).

In [8]:
print("[INFO] Menjalankan validasi kualitas data...")

# 1) Tidak ada duplikasi primary key
assert df_master_analisis["order_id"].is_unique, "Fatal: Terdeteksi order_id duplikat!"
print("  [OK] Primary key order_id unik")

# 2) total_amount tidak negatif
assert (df_master_analisis["total_amount"] >= 0).all(), "Fatal: Ada nilai transaksi (total_amount) negatif!"
print("  [OK] Tidak ada total_amount negatif")

# 3) Tidak ada tanggal pesanan di masa depan
hari_ini = pd.Timestamp.today().normalize()
assert (df_master_analisis["order_date"] <= hari_ini).all(), (
    f"Fatal: Ada order_date melebihi hari ini ({hari_ini.date()})!")
print(f"  [OK] Tidak ada future date (tanggal terbaru: {df_master_analisis['order_date'].max().date()}, hari ini: {hari_ini.date()})")

# --- Pemeriksaan tambahan ---
assert df_master_analisis["order_id"].notna().all(), "Fatal: order_id mengandung null!"
assert df_master_analisis["total_amount"].notna().all(), "Fatal: total_amount mengandung null!"
assert pd.api.types.is_datetime64_any_dtype(df_master_analisis["order_date"]), "Fatal: order_date bukan datetime!"
print("  [OK] Pemeriksaan tambahan (null key, null amount, tipe datetime) lulus")

print("[SUCCESS] Seluruh validasi data lulus!")

[INFO] Menjalankan validasi kualitas data...
  [OK] Primary key order_id unik
  [OK] Tidak ada total_amount negatif
  [OK] Tidak ada future date (tanggal terbaru: 2026-09-29, hari ini: 2026-10-02)
  [OK] Pemeriksaan tambahan (null key, null amount, tipe datetime) lulus
[SUCCESS] Seluruh validasi data lulus!


## Kesimpulan
- Tiga sumber data heterogen (CSV POS, SQLite CRM, REST API JSON bersarang) berhasil diintegrasikan menjadi `df_master_analisis` tanpa kehilangan maupun duplikasi baris.
- Penggunaan `LEFT JOIN` menjaga seluruh transaksi berkupon tetap terhitung sehingga angka kampanye tidak bias.
- Kamus data otomatis tersimpan di `kamus_data_hasil.csv`, dan seluruh *automated assertion* lulus sehingga dataset layak dipakai BI Analyst.